# SEC Form 13F position-universe audit

This notebook audits the position universe produced by `scripts/04_build_sec_13f_position_universe.py` across the complete sample.

It examines:

1. consistency between selected filings and the filing audit;
2. attrition from the sequential position filters;
3. filing-level row and value reconciliation;
4. uniqueness and validity of the final manager-quarter-CUSIP observations; and
5. quarterly universe coverage.

The notebook reads the saved outputs and does not repeat the construction logic.

## 1. Setup and outputs

In [1]:
import sys
from pathlib import Path

import duckdb
import pandas as pd
import pyarrow.parquet as pq


project_root = Path.cwd().resolve()

while not (project_root / "config" / "paths.yaml").is_file():
    if project_root.parent == project_root:
        raise FileNotFoundError("Could not locate the project root.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.configuration import get_paths_config


paths = get_paths_config()
sec_directory = paths.interim / "sec_13f"
universe_directory = sec_directory / "position_universe"

output_paths = {
    "manager_security_positions": (
        universe_directory / "manager_security_positions.parquet"
    ),
    "position_filter_audit": universe_directory / "position_filter_audit.parquet",
    "filing_audit": universe_directory / "filing_audit.parquet",
}
selected_filings_path = (
    sec_directory / "filing_selection" / "selected_filings.parquet"
)

for path in (*output_paths.values(), selected_filings_path):
    if not path.is_file():
        raise FileNotFoundError(f"Required input not found: {path}")

position_path = output_paths["manager_security_positions"]
filter_audit = pd.read_parquet(output_paths["position_filter_audit"])
filing_audit = pd.read_parquet(output_paths["filing_audit"])
selected_filings = pd.read_parquet(
    selected_filings_path,
    columns=["ACCESSION_NUMBER"],
)
connection = duckdb.connect(database=":memory:")

inventory = pd.DataFrame(
    {
        "table": list(output_paths),
        "rows": [
            pq.ParquetFile(path).metadata.num_rows
            for path in output_paths.values()
        ],
    }
)

display(inventory.style.format({"rows": "{:,}"}))

,table,rows
0,manager_security_positions,"60,766,163"
1,position_filter_audit,364
2,filing_audit,"279,051"


## 2. Output consistency

The filing audit should contain exactly the accessions selected in the previous stage. The filter audit should contain one row for every report quarter and filter stage.

In [2]:
selected_accessions = set(selected_filings["ACCESSION_NUMBER"])
audited_accessions = set(filing_audit["ACCESSION_NUMBER"])

expected_stages = {
    "Selected filing rows",
    "Baseline-report rows",
    "Share-denominated rows",
    "Non-option rows",
    "Valid nine-character CUSIP rows",
    "Numeric-value rows",
    "Positive-value baseline rows",
}

period_stage_duplicates = filter_audit.duplicated(
    ["report_period", "stage"]
).sum()
period_stage_counts = filter_audit.groupby("report_period")["stage"].nunique()

output_checks = pd.DataFrame(
    {
        "check": [
            "Duplicate accessions in the filing audit",
            "Mismatch between selected and audited accessions",
            "Unexpected or missing filter stages",
            "Duplicate report-period/filter-stage rows",
            "Report periods without all filter stages",
        ],
        "violations": [
            filing_audit["ACCESSION_NUMBER"].duplicated().sum(),
            len(selected_accessions ^ audited_accessions),
            len(set(filter_audit["stage"]) ^ expected_stages),
            period_stage_duplicates,
            period_stage_counts.ne(len(expected_stages)).sum(),
        ],
    }
)

display(output_checks.style.format({"violations": "{:,}"}))
print()
print(f"Total output-consistency violations: {output_checks['violations'].sum():,}")

,check,violations
0,Duplicate accessions in the filing audit,0
1,Mismatch between selected and audited accessions,0
2,Unexpected or missing filter stages,0
3,Duplicate report-period/filter-stage rows,0
4,Report periods without all filter stages,0



Total output-consistency violations: 0


## 3. Sequential position filters

The stages are cumulative. Retention relative to the preceding row identifies which methodological restriction removes each set of positions.

In [3]:
stage_order = filter_audit["stage"].drop_duplicates().tolist()

filter_summary = (
    filter_audit.groupby("stage", sort=False)
    .agg(
        rows=("rows", "sum"),
        manager_periods=("manager_periods", "sum"),
        value_usd=("value_usd", "sum"),
    )
    .reindex(stage_order)
)

filter_summary["row_retention_from_start_percent"] = (
    filter_summary["rows"] / filter_summary["rows"].iloc[0] * 100
)
filter_summary["row_retention_from_previous_percent"] = (
    filter_summary["rows"] / filter_summary["rows"].shift(1) * 100
).fillna(100)
filter_summary["value_trillions_usd"] = (
    filter_summary["value_usd"] / 1_000_000_000_000
)

display(
    filter_summary[
        [
            "rows",
            "manager_periods",
            "row_retention_from_start_percent",
            "row_retention_from_previous_percent",
            "value_trillions_usd",
        ]
    ].style.format(
        {
            "rows": "{:,}",
            "manager_periods": "{:,}",
            "row_retention_from_start_percent": "{:.2f}",
            "row_retention_from_previous_percent": "{:.2f}",
            "value_trillions_usd": "{:.2f}",
        }
    )
)

,rows,manager_periods,row_retention_from_start_percent,row_retention_from_previous_percent,value_trillions_usd
stage,,,,,
Selected filing rows,"104,901,260","278,609",100.00,100.00,3828.98
Baseline-report rows,"87,893,748","264,556",83.79,83.79,3048.09
Share-denominated rows,"87,158,106","264,193",83.09,99.16,3022.31
Non-option rows,"81,640,635","263,985",77.83,93.67,2885.30
Valid nine-character CUSIP rows,"81,640,635","263,985",77.83,100.00,2885.30
Numeric-value rows,"81,640,635","263,985",77.83,100.00,2885.30
Positive-value baseline rows,"80,192,301","260,473",76.45,98.23,2885.30


## 4. Filing-level reconciliation and value units

Reconciliation compares all selected information-table rows with the SEC summary declaration before analytical position filters are applied. Missing or conflicting source declarations remain visible rather than being silently corrected.

In [4]:
reconciliation_summary = pd.DataFrame(
    {
        "check": [
            "Selected filings audited",
            "Filings without a summary row",
            "Filings without observed position rows",
            "Position-count mismatches",
            "Position-count comparisons unavailable",
            "Value mismatches",
            "Value comparisons unavailable",
            "Filings containing non-numeric position values",
        ],
        "count": [
            len(filing_audit),
            (~filing_audit["has_summary_row"]).sum(),
            filing_audit["observed_position_rows"].eq(0).sum(),
            filing_audit["position_count_matches_summary"].eq(False).sum(),
            filing_audit["position_count_matches_summary"].isna().sum(),
            filing_audit["value_matches_summary"].eq(False).sum(),
            filing_audit["value_matches_summary"].isna().sum(),
            filing_audit["invalid_value_rows"].gt(0).sum(),
        ],
    }
)

unit_summary = (
    filing_audit.groupby("value_unit_classification", as_index=False)
    .agg(
        filings=("ACCESSION_NUMBER", "size"),
        observed_value_usd=("observed_value_usd", "sum"),
        value_mismatches=("value_matches_summary", lambda values: values.eq(False).sum()),
    )
    .sort_values("filings", ascending=False)
)
unit_summary["observed_value_share_percent"] = (
    unit_summary["observed_value_usd"]
    / unit_summary["observed_value_usd"].sum()
    * 100
)

display(reconciliation_summary.style.format({"count": "{:,}"}))
display(
    unit_summary.style.format(
        {
            "filings": "{:,}",
            "observed_value_usd": "${:,.0f}",
            "value_mismatches": "{:,}",
            "observed_value_share_percent": "{:.2f}",
        }
    )
)

relative_difference = (
    filing_audit["value_difference_usd"].abs()
    / filing_audit["declared_value_usd"].abs().replace(0, pd.NA)
)
materiality = pd.Series(
    "Comparison unavailable",
    index=filing_audit.index,
    dtype="string",
)
is_mismatch = filing_audit["value_matches_summary"].eq(False)
materiality.loc[filing_audit["value_matches_summary"].eq(True)] = "Exact match"
materiality.loc[is_mismatch & relative_difference.le(0.0001)] = (
    "Difference at most 0.01%"
)
materiality.loc[
    is_mismatch & relative_difference.gt(0.0001) & relative_difference.le(0.01)
] = "Difference above 0.01% and at most 1%"
materiality.loc[
    is_mismatch & relative_difference.gt(0.01) & relative_difference.le(1)
] = "Difference above 1% and at most 100%"
materiality.loc[is_mismatch & relative_difference.gt(1)] = "Difference above 100%"
materiality.loc[is_mismatch & relative_difference.isna()] = (
    "Relative difference unavailable"
)

materiality_order = [
    "Exact match",
    "Difference at most 0.01%",
    "Difference above 0.01% and at most 1%",
    "Difference above 1% and at most 100%",
    "Difference above 100%",
    "Relative difference unavailable",
    "Comparison unavailable",
]
materiality_summary = (
    materiality.value_counts()
    .reindex(materiality_order, fill_value=0)
    .rename_axis("value_reconciliation")
    .reset_index(name="filings")
)

print()
print("Value-reconciliation materiality")
display(materiality_summary.style.format({"filings": "{:,}"}))

value_mismatches = filing_audit.loc[
    filing_audit["value_matches_summary"].eq(False),
    [
        "ACCESSION_NUMBER",
        "report_period",
        "value_unit_classification",
        "declared_value_usd",
        "observed_value_usd",
        "value_difference_usd",
    ],
].copy()

if not value_mismatches.empty:
    value_mismatches["absolute_difference_usd"] = value_mismatches[
        "value_difference_usd"
    ].abs()
    print()
    print("Largest value-reconciliation differences")
    display(
        value_mismatches.nlargest(10, "absolute_difference_usd")
        .reset_index(drop=True)
    )

,check,count
0,Selected filings audited,"279,051"
1,Filings without a summary row,16
2,Filings without observed position rows,47
3,Position-count mismatches,"12,034"
4,Position-count comparisons unavailable,18
5,Value mismatches,"38,915"
6,Value comparisons unavailable,55
7,Filings containing non-numeric position values,0


,value_unit_classification,filings,observed_value_usd,value_mismatches,observed_value_share_percent
3,PRE_2023_THOUSANDS,"165,262","$2,634,728,809,988,000","26,697",68.81
0,POST_2023_DOLLARS,"100,294","$697,356,398,282,829","2,541",18.21
4,PRE_2023_UNRESOLVED,"9,316","$475,004,963,359,000","6,796",12.41
1,POST_2023_UNRESOLVED,"2,798","$12,690,831,373,754","1,622",0.33
2,PRE_2023_INFORMATION_TABLE_DOLLARS,"1,381","$9,200,889,633,785","1,259",0.24



Value-reconciliation materiality


,value_reconciliation,filings
0,Exact match,"240,081"
1,Difference at most 0.01%,"30,497"
2,Difference above 0.01% and at most 1%,"2,702"
3,Difference above 1% and at most 100%,"3,788"
4,Difference above 100%,"1,600"
5,Relative difference unavailable,328
6,Comparison unavailable,55



Largest value-reconciliation differences


,ACCESSION_NUMBER,report_period,value_unit_classification,declared_value_usd,observed_value_usd,value_difference_usd,absolute_difference_usd
0,0000019617-14-000465,2014-09-30,PRE_2023_UNRESOLVED,420335463171000,420679489000,-419914783682000,419914783682000
1,0000019617-14-000425,2014-06-30,PRE_2023_UNRESOLVED,393880169829000,393880237000,-393486289592000,393486289592000
2,0000019617-14-000357,2014-03-31,PRE_2023_UNRESOLVED,362466811351000,362466766000,-362104344585000,362104344585000
3,0000019617-14-000264,2013-12-31,PRE_2023_UNRESOLVED,349208534545000,349208531000,-348859326014000,348859326014000
4,0000019617-13-000398,2013-09-30,PRE_2023_UNRESOLVED,337448845555000,337448794000,-337111396761000,337111396761000
5,0000019617-13-000358,2013-06-30,PRE_2023_UNRESOLVED,313748451004000,313748534000,-313434702470000,313434702470000
6,0001538383-18-000004,2018-09-30,PRE_2023_UNRESOLVED,312769808000000,312769808000,-312457038192000,312457038192000
7,0000861177-21-000012,2021-09-30,PRE_2023_UNRESOLVED,232747600000,241281043035000,241048295435000,241048295435000
8,0000923116-14-000001,2013-12-31,PRE_2023_UNRESOLVED,240623000000000,240623000,-240622759377000,240622759377000
9,0000923116-14-000005,2014-09-30,PRE_2023_UNRESOLVED,210104000000000,210104000,-210103789896000,210103789896000


## 5. Final-position integrity

The final table should contain one positive-valued observation per manager, report quarter, and nine-character CUSIP. The large position table is queried directly from Parquet rather than loaded into pandas.

In [5]:
position_metrics = connection.execute(
    '''
    SELECT
        COUNT(*) AS positions,
        COUNT(*) - COUNT(DISTINCT (CIK, report_period, CUSIP))
            AS duplicate_keys,
        COUNT(*) FILTER (
            WHERE CIK IS NULL OR report_period IS NULL OR CUSIP IS NULL
        ) AS missing_key_rows,
        COUNT(*) FILTER (WHERE length(CUSIP) != 9) AS invalid_cusip_rows,
        COUNT(*) FILTER (WHERE position_value_usd <= 0) AS nonpositive_values,
        COUNT(*) FILTER (WHERE raw_position_rows < 1) AS invalid_source_counts,
        COUNT(*) FILTER (WHERE NAMEOFISSUER IS NULL) AS missing_issuer_labels,
        COUNT(*) FILTER (WHERE TITLEOFCLASS IS NULL) AS missing_class_labels,
        COUNT(DISTINCT report_period) AS report_quarters,
        COUNT(DISTINCT CIK) AS managers,
        COUNT(DISTINCT CUSIP) AS cusips
    FROM read_parquet(?)
    ''',
    [str(position_path)],
).fetchdf()

position_checks = pd.DataFrame(
    {
        "check": [
            "Duplicate manager-quarter-CUSIP keys",
            "Rows with a missing key component",
            "Rows without a nine-character CUSIP",
            "Rows with a non-positive value",
            "Rows with an invalid source-row count",
            "Rows with a missing issuer label",
            "Rows with a missing security-class label",
        ],
        "violations": [
            position_metrics.at[0, "duplicate_keys"],
            position_metrics.at[0, "missing_key_rows"],
            position_metrics.at[0, "invalid_cusip_rows"],
            position_metrics.at[0, "nonpositive_values"],
            position_metrics.at[0, "invalid_source_counts"],
            position_metrics.at[0, "missing_issuer_labels"],
            position_metrics.at[0, "missing_class_labels"],
        ],
    }
)

display(position_checks.style.format({"violations": "{:,}"}))

print()
print(f"Positions: {position_metrics.at[0, 'positions']:,}")
print(f"Managers: {position_metrics.at[0, 'managers']:,}")
print(f"CUSIPs: {position_metrics.at[0, 'cusips']:,}")
print(f"Report quarters: {position_metrics.at[0, 'report_quarters']:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,check,violations
0,Duplicate manager-quarter-CUSIP keys,0
1,Rows with a missing key component,0
2,Rows without a nine-character CUSIP,0
3,Rows with a non-positive value,0
4,Rows with an invalid source-row count,0
5,Rows with a missing issuer label,0
6,Rows with a missing security-class label,0



Positions: 60,766,163
Managers: 12,575
CUSIPs: 99,889
Report quarters: 52


## 6. Quarterly coverage

Quarterly counts describe the changing scale of the manager-security universe before mapping CUSIPs to CRSP securities.

In [6]:
quarterly_coverage = connection.execute(
    '''
    SELECT
        report_period,
        COUNT(DISTINCT CIK) AS managers,
        COUNT(DISTINCT CUSIP) AS cusips,
        COUNT(*) AS manager_security_positions,
        SUM(position_value_usd) AS reported_value_usd
    FROM read_parquet(?)
    GROUP BY report_period
    ORDER BY report_period
    ''',
    [str(position_path)],
).fetchdf()

connection.close()

display(
    quarterly_coverage.style.format(
        {
            "managers": "{:,}",
            "cusips": "{:,}",
            "manager_security_positions": "{:,}",
            "reported_value_usd": "${:,.0f}",
        }
    )
)

coverage_range = pd.DataFrame(
    {
        "measure": ["Managers", "CUSIPs", "Manager-security positions"],
        "minimum": [
            quarterly_coverage["managers"].min(),
            quarterly_coverage["cusips"].min(),
            quarterly_coverage["manager_security_positions"].min(),
        ],
        "maximum": [
            quarterly_coverage["managers"].max(),
            quarterly_coverage["cusips"].max(),
            quarterly_coverage["manager_security_positions"].max(),
        ],
    }
)

print()
display(coverage_range.style.format({"minimum": "{:,}", "maximum": "{:,}"}))

,report_period,managers,cusips,manager_security_positions,reported_value_usd
0,2013-06-30 00:00:00,"2,785","17,960","649,189","$99,796,713,834,364"
1,2013-09-30 00:00:00,"2,946","19,496","712,730","$102,595,245,590,594"
2,2013-12-31 00:00:00,"3,275","20,120","758,241","$154,287,431,186,394"
3,2014-03-31 00:00:00,"3,323","19,475","785,439","$144,680,219,007,726"
4,2014-06-30 00:00:00,"3,280","19,848","798,323","$195,453,940,966,555"
5,2014-09-30 00:00:00,"3,327","19,976","803,349","$131,884,124,709,481"
6,2014-12-31 00:00:00,"3,576","19,348","833,278","$75,106,448,038,398"
7,2015-03-31 00:00:00,"3,640","20,390","851,330","$56,252,345,183,154"
8,2015-06-30 00:00:00,"3,630","21,068","856,173","$75,238,452,086,689"
9,2015-09-30 00:00:00,"3,658","19,296","853,403","$52,365,515,808,463"


,measure,minimum,maximum
0,Managers,"2,785","8,087"
1,CUSIPs,"16,281","22,678"
2,Manager-security positions,"649,189","2,031,484"


## 7. Interpretation

- **The saved outputs are mutually consistent.** The filing audit covers exactly the 279,051 selected accessions, and all 52 report quarters contain the seven expected filter stages without duplicate rows.
- **The filters produce a structurally clean position universe.** The process retains 80,192,301 of 104,901,260 selected information-table rows before aggregating them into 60,766,163 unique manager-quarter-CUSIP positions. All final keys are unique and complete; CUSIPs have nine characters; values are positive; and the two missing issuer names found in Notebook 1 do not leave missing canonical labels in the final table.
- **Most attrition is intentional.** Restricting the baseline to holdings reports retains 83.79% of selected rows. Removing non-share and option positions produces most of the remaining reduction. The valid-CUSIP and numeric-value stages remove no additional rows, while the positive-value condition retains 98.23% of the preceding stage.
- **Some selected filings contain no usable portfolio.** Forty-seven selected filings have no observed information-table rows and therefore do not enter the position universe. Sixteen selected filings lack a summary-page row. These cases remain documented in the filing audit.
- **Exact reconciliation failures vary in importance.** There are 12,034 position-count mismatches and 38,915 value mismatches. Of the value mismatches, 30,497 are no larger than 0.01% of the declared total, but 5,388 exceed 1% and 1,600 exceed 100%. In addition, 12,114 filings have unresolved value-unit classifications; under the current conversion they account for 12.74% of observed pre-filter value. Absolute-dollar analyses should therefore retain these flags and include a sensitivity treatment for unresolved units rather than treating every mismatch as harmless.
- **Coverage expands materially over time.** The quarterly universe ranges from 2,785 to 8,087 managers and from 649,189 to 2,031,484 manager-security positions. Later network comparisons should account for this changing cross-sectional coverage.

The identifiers and row structure are suitable for the next CUSIP-to-CRSP mapping stage. The unresolved value-unit cases do not prevent identifier mapping, but they must be addressed before interpreting absolute holding values or value-dependent results.